# 简单数据分析 Agent：用自然语言查询 DataFrame

## 概述
这一节我们实现一个数据分析 Agent：你用自然语言提问，它会对一个 `pandas.DataFrame` 做必要的计算，然后给出回答。

## 动机
很多数据分析门槛来自“必须会写代码”。把“问题 → 计算 → 解释”串起来后，非技术用户也能用自然语言探索数据。

## 关键组件
- **语言模型**：理解问题、决定要算什么、组织回答
- **DataFrame（pandas）**：承载结构化数据与计算
- **工具调用**：把“要算的部分”交给工具（Python/pandas），把结果以 `ToolMessage` 形式返回
- **LangGraph**：用图来组织“模型 → 工具 → 模型”的循环

## 方法
1) 生成一份合成的汽车销售数据（DataFrame）
2) 定义一个 DataFrame 分析工具（能在 `df` 上执行少量 pandas 计算）
3) 用 LangGraph 组装一个最小的 tool-calling graph：`call_model → tools → call_model`
4) 用几条示例问题验证效果，并观察 `AIMessage/ToolMessage` 的完整信息


## 导入库并加载环境变量

In [1]:
from __future__ import annotations

import json
import os
from datetime import datetime, timedelta
from typing import Annotated, TypedDict

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode


load_dotenv("../.env")
np.random.seed(42)

## 生成示例数据

我们构造一个合成的汽车销售数据集（1000 行），字段包括：日期、品牌、车型、价格、里程、油耗、销售员等。

In [2]:
n_rows = 1000

# 日期
start_date = datetime(2022, 1, 1)
dates = [start_date + timedelta(days=i) for i in range(n_rows)]

# 类别字段
makes = ["Toyota", "Honda", "Ford", "Chevrolet", "Nissan", "BMW", "Mercedes", "Audi", "Hyundai", "Kia"]
models = ["Sedan", "SUV", "Truck", "Hatchback", "Coupe", "Van"]
colors = ["Red", "Blue", "Black", "White", "Silver", "Gray", "Green"]
sales_people = ["Alice", "Bob", "Charlie", "David", "Eva"]

# 数据
data = {
    "Date": dates,
    "Make": np.random.choice(makes, n_rows),
    "Model": np.random.choice(models, n_rows),
    "Color": np.random.choice(colors, n_rows),
    "Year": np.random.randint(2015, 2023, n_rows),
    "Price": np.random.uniform(20000, 80000, n_rows).round(2),
    "Mileage": np.random.uniform(0, 100000, n_rows).round(0),
    "EngineSize": np.random.choice([1.6, 2.0, 2.5, 3.0, 3.5, 4.0], n_rows),
    "FuelEfficiency": np.random.uniform(20, 40, n_rows).round(1),
    "SalesPerson": np.random.choice(sales_people, n_rows),
}

df = pd.DataFrame(data).sort_values("Date")
df.head()

,Date,Make,Model,Color,Year,Price,Mileage,EngineSize,FuelEfficiency,SalesPerson
0,2022-01-01,Mercedes,Sedan,Green,2022,57952.65,5522.0,2.0,24.7,Alice
1,2022-01-02,Chevrolet,Hatchback,Red,2021,58668.22,94238.0,1.6,26.2,Bob
2,2022-01-03,Audi,Truck,White,2019,69187.87,7482.0,2.0,28.0,David
3,2022-01-04,Nissan,Hatchback,Black,2016,40004.44,43846.0,3.5,24.8,David
4,2022-01-05,Mercedes,Hatchback,Red,2016,63983.07,52988.0,2.5,24.1,Alice


In [3]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Date            1000 non-null   datetime64[ns]
 1   Make            1000 non-null   object        
 2   Model           1000 non-null   object        
 3   Color           1000 non-null   object        
 4   Year            1000 non-null   int64         
 5   Price           1000 non-null   float64       
 6   Mileage         1000 non-null   float64       
 7   EngineSize      1000 non-null   float64       
 8   FuelEfficiency  1000 non-null   float64       
 9   SalesPerson     1000 non-null   object        
dtypes: datetime64[ns](1), float64(4), int64(1), object(4)
memory usage: 78.3+ KB


,Date,Year,Price,Mileage,EngineSize,FuelEfficiency
count,1000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,2023-05-15 12:00:00,2018.445000,51145.360800,48484.643000,2.744500,29.688500
min,2022-01-01 00:00:00,2015.000000,20026.570000,19.000000,1.600000,20.000000
25%,2022-09-07 18:00:00,2017.000000,36859.940000,23191.500000,2.000000,24.500000
50%,2023-05-15 12:00:00,2018.000000,52215.155000,47506.000000,2.500000,29.700000
75%,2024-01-20 06:00:00,2020.000000,65741.147500,73880.250000,3.500000,34.700000
max,2024-09-26 00:00:00,2022.000000,79972.640000,99762.000000,4.000000,40.000000
std,NaN,2.256117,17041.610861,29103.404593,0.839389,5.896316


## 定义 DataFrame 分析工具

我们给模型一个工具：它可以在当前 `df` 上做计算，并把结果以文本形式返回。

为了让示例更可控，这个工具只暴露一个入口：`pandas_query(code)`。


In [4]:
@tool
def pandas_query(code: str) -> str:
    """在 DataFrame `df` 上执行一段 pandas 代码，并返回结果文本。

约定：
- 你可以使用变量 `df`
- 如果你想返回一个值，请把最后一行写成一个表达式（例如 `df['Price'].mean()`）
"""

    # 只开放必要的对象
    local_vars = {"df": df, "pd": pd, "np": np}
    safe_builtins = {
        "len": len,
        "min": min,
        "max": max,
        "sum": sum,
        "round": round,
        "sorted": sorted,
        "list": list,
        "dict": dict,
        "set": set,
        "tuple": tuple,
        "int": int,
        "float": float,
        "str": str,
    }
    global_vars = {"__builtins__": safe_builtins}

    # 支持两种写法：
    # 1) 单表达式：直接 eval 并返回
    # 2) 多行：先 exec，再尝试把最后一行当表达式 eval
    code_str = (code or "").strip()
    if not code_str:
        return "(empty code)"

    lines = [ln for ln in code_str.splitlines() if ln.strip()]
    if len(lines) == 1:
        try:
            out = eval(lines[0], global_vars, local_vars)
            if isinstance(out, (pd.DataFrame, pd.Series)):
                return out.to_string(max_rows=20)
            return repr(out)
        except Exception as e:
            return f"ERROR: {type(e).__name__}: {e}"

    prefix = "\n".join(lines[:-1])
    last = lines[-1]
    try:
        exec(prefix, global_vars, local_vars)
        out = eval(last, global_vars, local_vars)
        if isinstance(out, (pd.DataFrame, pd.Series)):
            return out.to_string(max_rows=20)
        return repr(out)
    except Exception:
        try:
            exec(code_str, global_vars, local_vars)
            return "OK"
        except Exception as e:
            return f"ERROR: {type(e).__name__}: {e}"

## 构建数据分析 Agent（LangGraph + tool calling）

这部分是核心：

- `call_model`：模型决定要不要调用工具（如果需要计算，就生成 tool_calls）
- `tools`：执行工具，返回 `ToolMessage`
- 再回到 `call_model`：模型基于工具结果生成最终回答


In [5]:
llm = ChatOpenAI(
    model="deepseek-v4-flash-0731",
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
    base_url=os.environ.get("DASHSCOPE_BASE_URL"),
    temperature=0,
)

tools = [pandas_query]
llm_with_tools = llm.bind_tools(tools)


class State(TypedDict):
    messages: Annotated[list, add_messages]


def call_model(state: State) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


tool_node = ToolNode(tools)


def should_continue(state: State):
    last = state["messages"][-1]
    tool_calls = getattr(last, "tool_calls", None) or []
    return "tools" if tool_calls else END


builder = StateGraph(State)
builder.add_node("call_model", call_model)
builder.add_node("tools", tool_node)
builder.add_edge(START, "call_model")
builder.add_conditional_edges("call_model", should_continue, {"tools": "tools", END: END})
builder.add_edge("tools", "call_model")

app = builder.compile()

## 提问函数 + 可观测输出

我们写一个 `ask_agent(question)`：

- 打印最终回答
- 同时把整段 `messages` 里每条消息的类型（Human/AI/Tool）打印出来
- 如果最后一条是 `AIMessage`，额外 dump 它的完整元信息


In [6]:
SYSTEM_PROMPT = (
    "你是一个数据分析 Agent。你可以使用一个名为 df 的 pandas DataFrame。"
    "当用户的问题需要任何统计、排序、筛选、聚合等计算时，先调用 pandas_query 工具完成计算，"
    "再基于工具返回结果给出清晰、简洁的回答。"
)


def ask_agent(question: str) -> None:
    result = app.invoke(
        {
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": question},
            ]
        }
    )
    messages = result["messages"]

    print("\nQuestion:", question)
    print("Answer:", messages[-1].content)
    print("\n--- Trace (messages) ---")
    for i, m in enumerate(messages):
        t = type(m).__name__
        content = getattr(m, "content", "")
        if isinstance(content, str) and len(content) > 200:
            content = content[:200] + " ..."
        name = getattr(m, "name", "")
        if t == "ToolMessage":
            print(f"[{i:02d}] {t}({name}): {content!r}")
        else:
            print(f"[{i:02d}] {t}: {content!r}")

    last = messages[-1]
    if type(last).__name__ == "AIMessage" and hasattr(last, "model_dump"):
        print("\n--- Last AIMessage dump ---")
        print(json.dumps(last.model_dump(), ensure_ascii=False, indent=2))

## 示例问题

下面这些问题会强制模型去调用 `pandas_query` 做计算（你会在 trace 里看到 tool_calls）。

In [7]:
ask_agent("What are the column names in this dataset?")
ask_agent("How many rows are in this dataset?")
ask_agent("What is the average price of cars sold?")
ask_agent("Which salesperson sold the most cars? Return the name only.")
ask_agent("Give me the top 3 car makes by average price.")


Question: What are the column names in this dataset?
Answer: The dataset contains the following columns:

1. **Date**
2. **Make**
3. **Model**
4. **Color**
5. **Year**
6. **Price**
7. **Mileage**
8. **EngineSize**
9. **FuelEfficiency**
10. **SalesPerson**

These appear to be car-related sales data. Let me know if you'd like any analysis on these!

--- Trace (messages) ---
[00] SystemMessage: '你是一个数据分析 Agent。你可以使用一个名为 df 的 pandas DataFrame。当用户的问题需要任何统计、排序、筛选、聚合等计算时，先调用 pandas_query 工具完成计算，再基于工具返回结果给出清晰、简洁的回答。'
[01] HumanMessage: 'What are the column names in this dataset?'
[02] AIMessage: ''
[03] ToolMessage(pandas_query): "['Date', 'Make', 'Model', 'Color', 'Year', 'Price', 'Mileage', 'EngineSize', 'FuelEfficiency', 'SalesPerson']"
[04] AIMessage: 'The dataset contains the following columns:\n\n1. **Date**\n2. **Make**\n3. **Model**\n4. **Color**\n5. **Year**\n6. **Price**\n7. **Mileage**\n8. **EngineSize**\n9. **FuelEfficiency**\n10. **SalesPerson**\n\nThes ...'

--- Last AIMessage d

## 检查理解（含答案）

1) **为什么这里需要 tool calling？**
- 答：模型本身不会“真的去算” DataFrame；当问题需要精确统计/排序/聚合时，把计算交给 `pandas_query`，结果以 `ToolMessage` 回来，然后模型再组织语言。

2) **`AIMessage` 和 `ToolMessage` 分别起什么作用？**
- 答：`AIMessage` 要么直接回答，要么产出 `tool_calls`（告诉运行时要调用哪个工具、参数是什么）；工具执行后返回 `ToolMessage`（承载真实计算结果）。

3) **为什么 graph 结构是 `call_model → tools → call_model`？**
- 答：第一次 `call_model` 用来“决定要不要算、怎么算”；`tools` 执行计算；第二次 `call_model` 用工具结果生成最终答案。

## 总结

- 你已经有了一个最小数据分析 Agent：自然语言提问 → 必要时调用 `pandas_query` → 输出回答。
- 当你需要更复杂的分析（多步推导、画图、写报告）时，可以把它们拆成更多工具，或者把图从“单节点循环”扩展成多节点工作流。